# TaskNest Support Ticket Router

Classify a customer ticket with an LLM, then send it to the matching support team using a LangGraph conditional edge. The team nodes are plain Python and do not call an LLM.

In [ ]:
!pip -q install langgraph langchain-openai

## 1. Configure the model and load the secret

Add `OPENAI_API_KEY` to Colab Secrets and enable notebook access to it. The key is loaded at runtime and is not included in this notebook. Change `MODEL_NAME` here if needed.

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

# Model name is defined once here so it is easy to change.
MODEL_NAME = "gpt-4o-mini"

# Read the API key from Colab Secrets at runtime.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Use deterministic model settings for consistent routing.
model = ChatOpenAI(model=MODEL_NAME, temperature=0)

## 2. Define the validated category and graph state

The classification schema allows only the three supported categories. The graph state contains exactly the ticket, category, team, and reply fields.

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(
        description="Classify the support ticket into billing, technical, or general."
    )

class TicketState(TypedDict):
    ticket: str
    category: str
    team: str
    reply: str

## 3. Create the classification node

The LLM categorizes the ticket by its meaning. This node returns only the category update.

In [ ]:
classifier = model.with_structured_output(TicketClassification)

def classify(state: TicketState) -> dict:
    """Choose a support category for the original ticket."""
    instructions = (
        "Classify the customer support ticket into exactly one category. "
        "Use billing for payments, invoices, refunds, or subscriptions; "
        "technical for bugs, crashes, error messages, or login problems; "
        "and general for feature ideas, general questions, or thank-you notes."
    )
    result = classifier.invoke([
        ("system", instructions),
        ("human", state["ticket"]),
    ])
    return {"category": result.category}

## 4. Route and handle the ticket

The router reads only the category from state. Each team node is plain Python: it sets a team name and a fixed confirmation reply.

In [ ]:
def route_ticket(state: TicketState) -> str:
    """Return a destination key using the existing category only."""
    return state["category"]

def billing_team(state: TicketState) -> dict:
    return {
        "team": "Billing Team",
        "reply": "Hello! Your billing inquiry has been received and forwarded to the Billing team.",
    }

def tech_team(state: TicketState) -> dict:
    return {
        "team": "Tech Team",
        "reply": "Hello! Your technical issue has been logged and sent to the Tech support team.",
    }

def general_team(state: TicketState) -> dict:
    return {
        "team": "General Team",
        "reply": "Hello! Thank you for reaching out. Your message has been sent to our General support team.",
    }

## 5. Build and visualize the graph

The workflow starts with classification, follows one conditional edge to a team, and then ends.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph

builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing", billing_team)
builder.add_node("technical", tech_team)
builder.add_node("general", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_ticket,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)

graph = builder.compile()

# Render the workflow diagram in the notebook.
display(Image(graph.get_graph().draw_mermaid_png()))

## 6. Route a sample ticket

Replace the sample text with a customer ticket. The final state shows its category, assigned team, and confirmation reply.

In [ ]:
ticket_text = "I was charged twice for my monthly subscription."

result = graph.invoke({
    "ticket": ticket_text,
    "category": "",
    "team": "",
    "reply": "",
})

print("Category:", result["category"])
print("Team:", result["team"])
print("Reply:", result["reply"])